In [1]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")

y_train = pd.read_csv(
    "../data/y_train.csv"
).squeeze()

In [2]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [3]:
from sklearn.linear_model import Ridge, Lasso
from src.evaluate import rmse_cv

In [4]:
model_ridge = Ridge(alpha=10.0)
model_lasso = Lasso(alpha=0.001)

ridge_rmse = rmse_cv(
    model_ridge,
    X_train,
    y_train
).mean()

lasso_rmse = rmse_cv(
    model_lasso,
    X_train,
    y_train
).mean()

print(f"Ridge RMSE: {ridge_rmse:.5f}")
print(f"Lasso RMSE: {lasso_rmse:.5f}")

Ridge RMSE: 0.11525
Lasso RMSE: 0.11359


In [5]:
# Linear Regression RMSE表格

import pandas as pd

result = pd.DataFrame({
    "Model":[
        "Ridge",
        "Lasso"
    ],
    "RMSE":[
        ridge_rmse,
        lasso_rmse
    ]
})

result

,Model,RMSE
0,Ridge,0.115253
1,Lasso,0.113587


In [6]:
result.to_csv(
    "../data/linear_regression_result.csv",
    index=False
)

In [7]:
# Lasso Submission

import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# 1. 讀取 preprocessing.ipynb 已經輸出的同一組資料
X_train_lasso = pd.read_csv("../data/X_train.csv")
y_train_lasso = pd.read_csv("../data/y_train.csv").squeeze("columns")
X_test_lasso = pd.read_csv("../data/X_test.csv")
test_ids = pd.read_csv("../data/test.csv", usecols=["Id"])["Id"]

# 2. 檢查 train 特徵與 target 筆數必須一致
print("X_train:", X_train_lasso.shape)
print("y_train:", y_train_lasso.shape)
print("X_test:", X_test_lasso.shape)

assert len(X_train_lasso) == len(y_train_lasso), "X_train 和 y_train 筆數不一致"
assert list(X_train_lasso.columns) == list(X_test_lasso.columns), "X_train 和 X_test 欄位不一致"

# 3. Lasso 對 feature scale 敏感，所以只用 train fit scaler，再 transform test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_lasso)
X_test_scaled = scaler.transform(X_test_lasso)

# 4. 訓練 Lasso；y_train.csv 已經是 log1p(SalePrice)
lasso = Lasso(alpha=0.0005, max_iter=10000)
lasso.fit(X_train_scaled, y_train_lasso)

# 5. 預測後轉回原本 SalePrice 尺度
preds_log = lasso.predict(X_test_scaled)
preds = np.expm1(preds_log)

# 6. 建立 Kaggle submission
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": preds
})

submission.to_csv(
    "../submissions/linear_submission.csv",
    index=False
)
submission.head()


X_train: (1458, 309)
y_train: (1458,)
X_test: (1459, 309)


PermissionError: [Errno 13] Permission denied: '../submissions/linear_submission.csv'